In [ ]:
import numpy as np
import scipy as sc
import matplotlib.pyplot as plt
import h5py
import jax.numpy as jnp
from jax import vmap
from jax_sph.jax_md import space
from matplotlib.colors import LinearSegmentedColormap
#cmap = LinearSegmentedColormap.from_list('own', ['black', '#615F5A', "#9D7204", "#AB9040", "#01AC76", "#2ADBB5", "#86D0DA", 'white'], 512)
#cmap = LinearSegmentedColormap.from_list('own', ["#55B0E1", "#76739B", "#DF81C3", "#D88E2C", "#EEB751", "#E9D773", 'white'], 512)
cmap = LinearSegmentedColormap.from_list('own', ['black', "#545046", "#8C670A", "#A88828",  "#939336", "#01AC57", "#2ADB77", "#86DA9E", 'white'], 512)

In [ ]:
c_data = np.linspace(0, 1, 512)[None, :]
fig, ax = plt.subplots(figsize=(5, 1))
ax.imshow(c_data, cmap=cmap, aspect='auto')
ax.axis('off')
fig.tight_layout()
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
fig.savefig('colorbar.png', dpi=100)

In [ ]:
path = '/local/disk1/datasets/dataset_kolm/2D_KOLM_4096_20kevery10/test.h5'
data = h5py.File(path, 'r')
r0 = np.asarray(data["/00001/position"])[200]
u0 = np.asarray(data["/00001/u"])[200]
r1 = np.asarray(data["/00001/position"])[201]
u1 = np.asarray(data["/00001/u"])[201]
r2 = np.asarray(data["/00001/position"])[202]
u30 = np.asarray(data["/00001/u"])[230]
r30 = np.asarray(data["/00001/position"])[230]
r31 = np.asarray(data["/00001/position"])[231]
data.close()

discplacement_fn, _ = space.periodic(side=2 * np.pi * np.ones(2))

dt = 0.005
v0 = vmap(discplacement_fn, in_axes=(0, 0))(r1, r0)
v0 /= dt
v1 = vmap(discplacement_fn, in_axes=(0, 0))(r2, r1)
v1 /= dt
v30 = vmap(discplacement_fn, in_axes=(0, 0))(r31, r30)
v30 /= dt

In [ ]:
fac = 5
vmin = 1.8
vmax = 3.7

xwv1 = np.linspace(-0.03, 2 * np.pi / fac, 256)
xwv1 = np.concatenate((xwv1, np.ones(50) * 2 * np.pi / fac))
xwv1 = np.concatenate((xwv1, np.linspace(2 * np.pi / fac, -0.03, 256)))
xwv1 = np.concatenate((xwv1, np.zeros(50) - 0.03))

ywv1 = 0.02* np.sin(10 * np.linspace(-0.03, 2 * np.pi / fac, 256)) + 2 * np.pi / fac - 0.03
ywv1 = np.concatenate((ywv1, np.linspace(ywv1[0], ywv1[0] + 0.06, 50)))
ywv1 = np.concatenate((ywv1, 0.02 * np.sin(10 * np.linspace(2 * np.pi / fac, -0.03, 256)) + 2 * np.pi / fac + 0.03))
ywv1 = np.concatenate((ywv1, np.linspace(ywv1[-1], ywv1[-1] - 0.06, 50)))

xwv2 = np.linspace(2 * np.pi / fac - 0.03, 2 * np.pi / fac + 0.03, 50)
xwv2 = np.concatenate((xwv2, np.sin(10 * np.linspace(0, 2 * np.pi / fac, 256)) * 0.02 + 2 * np.pi / fac + 0.03))
xwv2 = np.concatenate((xwv2, np.linspace(xwv2[0], xwv2[0] - 0.0, 50)))
xwv2 = np.concatenate((xwv2, np.sin(10 * np.linspace(2 * np.pi / fac, 0, 256)) * 0.02 + 2 * np.pi / fac - 0.03))

ywv2 = np.zeros(50) - 0.01
ywv2 = np.concatenate((ywv2, np.linspace(0, ywv1[255], 256)))
ywv2 = np.concatenate((ywv2, np.ones(50) * ywv1[255]))
ywv2 = np.concatenate((ywv2, np.linspace(ywv2[-1], 0, 256)))

xfix = np.array([xwv2[0]+0.001, xwv2[0]+0.2, xwv2[0]+0.2, xwv2[0]+0.001, xwv2[0]+0.001])
yfix = np.array([ywv2[611-256]-0.1, ywv2[611-256]-0.1, ywv2[611-256]+0.5, ywv2[611-256]+0.5, ywv2[611-256]-0.1]) 

xfix2 = np.array([xwv2[0]-0.01, xwv2[0]+0.2, xwv2[0]+0.2, xwv2[0]-0.01, xwv2[0]-0.01])
yfix2 = np.array([ywv2[611-256]+0.001, ywv2[611-256]+0.001, ywv2[611-256]+0.5, ywv2[611-256]+0.5, ywv2[611-256]+0.001])

leftx = np.zeros(256) + 0.003
lefty = np.linspace(0, 2*np.pi/fac -0.03, 256)

In [ ]:
fig2, ax = plt.subplots(figsize=(5, 5))
ax.scatter(r0[:, 0], r0[:, 1], c=np.linalg.norm(v0, axis=1), s=300, cmap=cmap, vmin=vmin, vmax=vmax)
ax.fill(xwv1, ywv1, color='white')
ax.plot(xwv1, ywv1, color='black', lw=4)
ax.fill(xwv2, ywv2, color='white')
ax.plot(xwv2, ywv2, color='black', lw=4)
ax.fill(xfix, yfix, color='white', zorder=10)
ax.fill(xfix2, yfix2, color='white', zorder=10)
ax.plot(leftx, lefty, color='black', lw=4)
ax.plot(lefty, leftx, color='black', lw=4)
ax.set_aspect('equal')
ax.axis('off')
ax.set_xlim(0, 2 * np.pi / fac)
ax.set_ylim(0, 2 * np.pi / fac)
fig2.subplots_adjust(left=0, right=1, top=1, bottom=0)
fig2.savefig('v0.png', dpi=100)

In [ ]:
fig2, ax = plt.subplots(figsize=(5, 5))
ax.scatter(r0[:, 0], r0[:, 1], c=np.linalg.norm(u0, axis=1), s=300, cmap=cmap, vmin=vmin, vmax=vmax)
ax.fill(xwv1, ywv1, color='white')
ax.plot(xwv1, ywv1, color='black', lw=4)
ax.fill(xwv2, ywv2, color='white')
ax.plot(xwv2, ywv2, color='black', lw=4)
ax.fill(xfix, yfix, color='white', zorder=10)
ax.fill(xfix2, yfix2, color='white', zorder=10)
ax.plot(leftx, lefty, color='black', lw=4)
ax.plot(lefty, leftx, color='black', lw=4)
ax.set_aspect('equal')
ax.axis('off')
ax.set_xlim(0, 2 * np.pi / fac)
ax.set_ylim(0, 2 * np.pi / fac)
fig2.subplots_adjust(left=0, right=1, top=1, bottom=0)
fig2.savefig('u0.png', dpi=100)

In [ ]:
vmax = 3.9

fig9, ax = plt.subplots(figsize=(5, 5))
ax.scatter(r30[:, 0], r30[:, 1], c=np.linalg.norm(v30, axis=1), s=300, cmap=cmap, vmin=vmin, vmax=vmax)
ax.fill(xwv1, ywv1, color='white')
ax.plot(xwv1, ywv1, color='black', lw=4)
ax.fill(xwv2, ywv2, color='white')
ax.plot(xwv2, ywv2, color='black', lw=4)
ax.fill(xfix, yfix, color='white', zorder=10)
ax.fill(xfix2, yfix2, color='white', zorder=10)
ax.plot(leftx, lefty, color='black', lw=4)
ax.plot(lefty, leftx, color='black', lw=4)
ax.set_aspect('equal')
ax.axis('off')
ax.set_xlim(0, 2 * np.pi / fac)
ax.set_ylim(0, 2 * np.pi / fac)
fig9.subplots_adjust(left=0, right=1, top=1, bottom=0)
fig9.savefig('v30.png', dpi=100)

In [ ]:
vmax = 3.9

fig10, ax = plt.subplots(figsize=(5, 5))
ax.scatter(r30[:, 0], r30[:, 1], c=np.linalg.norm(u30, axis=1), s=300, cmap=cmap, vmin=vmin, vmax=vmax)
ax.fill(xwv1, ywv1, color='white')
ax.plot(xwv1, ywv1, color='black', lw=4)
ax.fill(xwv2, ywv2, color='white')
ax.plot(xwv2, ywv2, color='black', lw=4)
ax.fill(xfix, yfix, color='white', zorder=10)
ax.fill(xfix2, yfix2, color='white', zorder=10)
ax.plot(leftx, lefty, color='black', lw=4)
ax.plot(lefty, leftx, color='black', lw=4)
ax.set_aspect('equal')
ax.axis('off')
ax.set_xlim(0, 2 * np.pi / fac)
ax.set_ylim(0, 2 * np.pi / fac)
fig10.subplots_adjust(left=0, right=1, top=1, bottom=0)
fig10.savefig('u30.png', dpi=100)